In [ ]:
from mrjob.job import MRJob
import re
import os
import uuid
import hashlib


class MRAnalizadorTXT(MRJob):
    
    JOBCONF = {
    'mapreduce.map.output.compress': False,
    'mapreduce.output.fileoutputformat.compress': False,
    'mapreduce.task.timeout': 600000
    }
    campos = {
    "nombre": r"Nombre:\s*([^\.]+)",
    "apellidos": r"Apellidos:\s*([^\.]+)",
    "CIPA": r"CIPA:\s*([^\.]+)",
    "nass": r"NASS:\s*([^\.]+)",
    "domicilio": r"Domicilio:\s*([^\.]+)",
    "Localidad_Provincia": r"Localidad/ Provincia:\s*([^\.]+)",
    "CP": r"CP:\s*(\d+)",
    "NHC": r"NHC:\s*(\d+)",
    "Fecha_nacimiento": r"Fecha de nacimiento:\s*([^\.]+)",
    "Pais": r"Pa[ií]s:\s*([^\.]+)",
    "Edad": r"Edad:\s*(\d+)",
    "Sexo": r"Sexo:\s*(\w)",
    "Fecha_Ingreso": r"Fecha de Ingreso:\s*([^\.]+)",
    "Especialidad": r"(?:Especialidad|Servicio):\s*([^\.]+)",
    "medico": r"M[eé]dico:\s*([^N]+)",
    "Numero_Colegiado": r"N[ºo]?Col:\s*([0-9\- ]+)",
    "Motivo_ingreso": r"Motivo de ingreso:\s*([^\.]*)",
    "Antecedentes": r"Antecedentes:\s*([^\.]*)",
    "Historia_Actual": r"Historia Actual:\s*(.*?)(?=Diagnostico|Tratamiento|$)",
    "Diagnostico_Principal": r"Diagn[oó]stico Principal:\s*(.*?)(?=Tratamiento:|Remitido por:|$)",
    "Exploracion_fisica_pruebas_complementarias": r"Exploraci[oó]n f[ií]sica y pruebas complementarias:\s*(.*?)(?=Diagn[oó]stico Principal:|Tratamiento:|Remitido por:|>
    "Tratamiento": r"Tratamiento:\s*(.*?)(?=Remitido|$)",
    "Informe_clinico_del_paciente": r"Informe clínico del paciente:\s*(.*?)(?=Diagnostico|Diagnóstico|Tratamiento|Remitido por:|$)"

    }
    contador = 0

    def mapper(self, _, line):

        import os

        file_path = os.environ.get("map_input_file", "unknown")

        if not line:
           return

        line = line.strip()

        if not line:
           return

        yield file_path, line


   
    def anonimizar(self, valor):

        if not valor:
           return ""

        return hashlib.sha256(valor.encode()).hexdigest()[:12]


    def reducer(self, key, values):

        texto = re.sub(r"\s+", " ", " ".join(values))

        datos = {}

        MRAnalizadorTXT.contador += 1
        datos["Id"] = MRAnalizadorTXT.contador

        for campo, patron in self.campos.items():
            m = re.search(patron, texto, re.IGNORECASE)
            datos[campo] = m.group(1).strip() if m else ""


        datos["nombre"] = "Paciente_" + str(datos["Id"])
        datos["apellidos"] = "Apellidos_" + str(self.anonimizar(datos["apellidos"]))
        datos["CIPA"] = self.anonimizar(datos["CIPA"])
        datos["domicilio"] = "Direccion_anonima"

        yield None, datos


if __name__ == "__main__":
    MRAnalizadorTXT.run()